In [1]:
import numpy as np
import pandas as pd
import os
import re

### Make sample table

In [ ]:
## make table that has the same categories for colony - sus, resis, sus-recover, sus-mortality 
# and has num of colonies that were sampled in 2019, may, and dec 2022 

In [2]:
os.getcwd()

'/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/CBC_metagenomics/all_sctld'

In [14]:
# upload sample_list 
with open("/scratch3/workspace/brooke_sienkiewicz_student_uml_edu-bel_sctld/genohublist24.txt", "r") as f:
    sample_list = [line.strip() for line in f] 

In [15]:
sample_list[0:5]

['062019_BEL_CBC_T3_25_PAST',
 '122022_BEL_CBC_T1_133_PSTR',
 '122022_BEL_CBC_T2_116_PSTR',
 '122022_BEL_CBC_T4_35_PSTR',
 '122022_BEL_CBC_T2_99_PSTR']

In [16]:
len(sample_list)

222

In [17]:
# match to sample metadata 

# load 
metadata=pd.read_csv('//project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/metadata/CBC_samples.csv')

#filter for UML samples (rna later or etoh) 
metadata=metadata[
    (metadata['Sample_type'] == 'Core_EtOH') |
    (metadata['Sample_type'] == 'Core_RNAlater')
]

#match list to tubelabel_species 
matched_metadata = metadata[metadata['Tubelabel_species'].isin(sample_list)]

In [20]:
# add colony ID - t# newtagnum species
matched_metadata = matched_metadata.copy()
matched_metadata['TransectNum_str'] = 'T' + matched_metadata['TransectNum'].astype(str)
matched_metadata['colony_id'] = matched_metadata[['TransectNum_str', 'NewTagNum', 'Species']].astype(str).agg('_'.join, axis=1)
matched_metadata.drop(columns='TransectNum_str', inplace=True)

In [21]:
matched_metadata.columns

Index(['Month_year', 'Country', 'Location', 'CollectionDate', 'Transect',
       'TransectNum', 'OldTagNum', 'NewTagNum', 'Species', 'Time_sampled',
       'Time_processed', 'Sample_type', 'SampleNum', 'Health_status',
       'Sampling_notes', 'Tubelabel_species', 'Sample_physical_location',
       'Extraction_physical_location', 'Date_sequenced', 'Notes', 'colony_id'],
      dtype='object')

In [22]:
matched_metadata[['Month_year','CollectionDate','Transect','TransectNum','NewTagNum',
                  'Species','SampleNum','Health_status','Sample_type','Tubelabel_species']]

,Month_year,CollectionDate,Transect,TransectNum,NewTagNum,Species,SampleNum,Health_status,Sample_type,Tubelabel_species
33,122022,12/4/22,CBC30N,1,22,OANN,120,Diseased_Margin,Core_EtOH,122022_BEL_CBC_T1_120_OANN
39,122022,12/2/22,CBC30N,1,22,OANN,136,Diseased_Tissue,Core_EtOH,122022_BEL_CBC_T1_136_OANN
139,52022,5/21/22,CBC30N,1,22,OANN,41,Healthy,Core_EtOH,052022_BEL_CBC_T1_41_OANN
256,122022,12/2/22,CBC30N,1,12,PSTR,122,Healthy,Core_EtOH,122022_BEL_CBC_T1_122_PSTR
261,122022,12/4/22,CBC30N,1,6,PSTR,132,Diseased_Tissue,Core_EtOH,122022_BEL_CBC_T1_132_PSTR
...,...,...,...,...,...,...,...,...,...,...
1129,62019,6/21/19,SR30N,2,68,PAST,2,Healthy,Core_EtOH,062019_BEL_CBC_T2_2_PAST
1141,62019,6/21/19,SR30N,2,59,MCAV,5,Healthy,Core_EtOH,062019_BEL_CBC_T2_5_MCAV
1144,62019,6/21/19,SR30N,2,330,MMEA,6,Healthy,Core_EtOH,062019_BEL_CBC_T2_6_MMEA
1146,62019,6/21/19,SR30N,2,344,MMEA,7,Healthy,Core_EtOH,062019_BEL_CBC_T2_7_MMEA


In [23]:
# make sample table 
# unique species, transect, health statuses, timepoint 
    # leaving monthyear as is for now 


In [24]:
matched_metadata['Month_year'].unique()

array([122022,  52022, 102019,  62019])

In [25]:
# combine 062019 and 102019? 

In [26]:
month_order = [62019, 102019, 52022, 122022]
matched_metadata['Month_year'] = pd.Categorical(
    matched_metadata['Month_year'], categories=month_order, ordered=True
)

In [27]:
summary_table = (
    matched_metadata
    .groupby(['Month_year', 'Transect', 'Species', 'Health_status'])
    .size()
    .reset_index(name='n')
    .pivot_table(index=['Month_year', 'Transect', 'Health_status'],
                 columns='Species',
                 values='n',
                 fill_value=0)
    .astype(int)
)

summary_table.index = summary_table.index.set_names(['Month_year', 'Transect', 'Health_status'])
summary_table = summary_table.reset_index()

summary_table = summary_table.set_index('Month_year')
summary_table.columns.name = None  # removes 'Species' label above columns

# remove rows with all 0s
species_cols = ['MCAV', 'MMEA', 'OANN', 'OFAV', 'PAST', 'PSTR']
summary_table = summary_table.loc[~(summary_table[species_cols] == 0).all(axis=1)]

/tmp/ipykernel_428721/3477100914.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['Month_year', 'Transect', 'Species', 'Health_status'])
/tmp/ipykernel_428721/3477100914.py:6: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(index=['Month_year', 'Transect', 'Health_status'],


In [28]:
summary_table

,Transect,Health_status,MCAV,MMEA,OANN,OFAV,PAST,PSTR
Month_year,,,,,,,,
62019,CBC30N,Healthy,9,2,0,0,5,0
62019,Lagoon,Healthy,8,5,0,0,9,0
62019,SR30N,Healthy,7,6,0,0,6,0
102019,CBC30N,Healthy,0,0,0,0,0,7
102019,Lagoon,Healthy,0,0,0,0,0,7
102019,SR30N,Healthy,0,0,0,0,0,7
52022,CBC30N,Diseased_Margin,3,0,0,0,0,1
52022,CBC30N,Diseased_Tissue,3,0,0,0,1,1
52022,CBC30N,Healthy,3,0,2,1,4,2


In [29]:
# just pre- and post- disease for each sp 

In [30]:
# group 2019s and 2022s
table = summary_table.copy()
table['Year'] = table.index.astype(str).str[-4:].astype(int)

In [31]:
# sort by location
condensed2 = (
    table
    .groupby(['Year', 'Transect','Health_status']) 
    .sum(numeric_only=True)
    .reset_index()
)
condensed2

,Year,Transect,Health_status,MCAV,MMEA,OANN,OFAV,PAST,PSTR
0,2019,CBC30N,Healthy,9,2,0,0,5,7
1,2019,Lagoon,Healthy,8,5,0,0,9,7
2,2019,SR30N,Healthy,7,6,0,0,6,7
3,2022,CBC30N,Diseased_Margin,3,0,1,0,1,2
4,2022,CBC30N,Diseased_Tissue,4,0,1,0,2,2
5,2022,CBC30N,Healthy,5,0,4,1,8,3
6,2022,CURLEW,Diseased_Margin,2,0,0,1,0,2
7,2022,CURLEW,Diseased_Tissue,2,0,0,1,0,2
8,2022,CURLEW,Healthy,3,0,0,3,0,3
9,2022,Lagoon,Diseased_Margin,4,0,0,0,2,0


In [32]:
# just pre and post per sp (no location)
condensed = (
    table
    .groupby(['Year', 'Health_status'])
    .sum(numeric_only=True)
    .reset_index()
)
condensed

,Year,Health_status,MCAV,MMEA,OANN,OFAV,PAST,PSTR
0,2019,Healthy,24,13,0,0,20,21
1,2022,Diseased_Margin,9,0,2,2,4,6
2,2022,Diseased_Tissue,11,0,3,2,5,6
3,2022,Healthy,24,0,8,15,24,21


In [33]:
## MCAV 
# 7 colonies died, 7 healthy were added in 2022 so number stayed the same 

In [34]:
## PAST 
# 20 original minus completely diseased and dead 
print(20-5)

# none added in 2022
# 2 sample points - 5 and 12/2022 so 30 healthy samples each
# minus 6 colonies that only got 1 2022 sample
print(30 - 6)

15
24


In [35]:
## PSTR 

#### investigate colony numbers
- ex: why are there 24 healthy mcav samples in 2019 AND 2022?
- this section has slightly more info, but was done before creating the above summary

##### **MCAV**

In [36]:
# crosscheck 2019 and 2022 colonies and samples

# start with mcav subset 
mcav=matched_metadata[matched_metadata['Species']=="MCAV"]
# list of all samples in 2019
meta_2019=mcav[
    (mcav['Month_year']==62019) |
    (mcav['Month_year']==102019)]
ids_2019=set(meta_2019['colony_id'].unique())
# list of all samples in 2022
meta_2022=mcav[
    (mcav['Month_year']==52022) |
    (mcav['Month_year']==122022)]
ids_2022=set(meta_2022['colony_id'].unique())
# do they match 
ids_2019 == ids_2022

False

In [37]:
# total mcav colonies 
print(len(mcav['colony_id'].unique()))

# colonies present in both years
print(len(ids_2019 & ids_2022))
# 17 shared

print(len(ids_2019))
# 24 initial

print(len(ids_2022))
# 7 added, 7 died 

31
17
24
24


In [38]:
# colonies not in 2022 
print('colonies not in 2022:',len(ids_2019 - ids_2022), 'colonies',
      ids_2019 - ids_2022)
# manually checking fate 
# all died in 052022

# new colonies in 2022 not present in 2019
print('new colonies in 2022 not present in 2019:',len(ids_2022 - ids_2019), 'colonies',
      ids_2022 - ids_2019)
# manually checking fate 
# all tagged in 122022

colonies not in 2022: 7 colonies {'T1_355_MCAV', 'T3_9_MCAV', 'T2_56_MCAV', 'T1_342_MCAV', 'T1_329_MCAV', 'T1_333_MCAV', 'T3_12_MCAV'}
new colonies in 2022 not present in 2019: 7 colonies {'T3_67_MCAV', 'T4_95_MCAV', 'T3_71_MCAV', 'T4_30_MCAV', 'T4_76_MCAV', 'T4_28_MCAV', 'T4_94_MCAV'}


In [39]:
# there just happen to be 7 mcav that died and 7 that were added

In [40]:
# check healthy 

# healthy 2019 samples 
healthy_mcav2019=meta_2019[meta_2019['Health_status']=='Healthy']
ids_h2019=set(healthy_mcav2019['colony_id'].unique())

# healthy 2022 samples 
healthy_mcav2022=meta_2022[meta_2022['Health_status']=='Healthy']
ids_h2022=set(healthy_mcav2022['colony_id'].unique())

# do they match 
ids_h2019 == ids_h2022

False

In [41]:
# colonies present in both years
print(len(ids_h2019 & ids_h2022))
# 11 stayed healthy 

print(len(ids_h2019))
# ALL started as healthy 

print(len(ids_h2022))

11
24
15


In [42]:
# healthy colonies not in 2022 
h2019disappeared=ids_h2019 - ids_h2022
print('healthy colonies that were not healthy in 2022:',len(ids_h2019 - ids_h2022), 'colonies',
      h2019disappeared)
# manually checking fate 
# died (5 colonies): 300s tags, t2_56, t3_9, t3_12
# got disease: t3_17, t1_15, t3_22, t1_8, t1_14, t3_15
# t1_8 seems to be the only one that has a disease sample from both 05 and 122022

# new healthy colonies in 2022 not present in 2019
print('new healthy colonies in 2022 not present in 2019:',len(ids_h2022 - ids_h2019), 'colonies',
      ids_h2022 - ids_h2019)
# manually checking fate 
# 4 of the 7 newly added were healthy in 2022 so this makes sense 

healthy colonies that were not healthy in 2022: 13 colonies {'T1_355_MCAV', 'T1_8_MCAV', 'T3_9_MCAV', 'T2_56_MCAV', 'T1_342_MCAV', 'T3_17_MCAV', 'T1_329_MCAV', 'T3_15_MCAV', 'T1_15_MCAV', 'T1_14_MCAV', 'T3_22_MCAV', 'T1_333_MCAV', 'T3_12_MCAV'}
new healthy colonies in 2022 not present in 2019: 4 colonies {'T4_30_MCAV', 'T3_71_MCAV', 'T4_76_MCAV', 'T4_28_MCAV'}


In [43]:
# weird ones: 
mcav[mcav['colony_id']=='T1_8_MCAV']

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
302,122022,BEL,CBC,12/2/22,CBC30N,1,NaN,8,MCAV,NaN,...,Core_EtOH,144,Diseased_Tissue,NaN,122022_BEL_CBC_T1_144_MCAV,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B26,NaN,NaN,T1_8_MCAV
641,52022,BEL,CBC,5/21/22,CBC30N,1,390,8,MCAV,NaN,...,Core_EtOH,12,Diseased_Margin,NaN,052022_BEL_CBC_T1_12_MCAV,Depleted_ UML_NARWHAL_R1_B3,DNA_extracted,sequenced,Only sand in tissue sample,T1_8_MCAV
643,52022,BEL,CBC,5/21/22,CBC30N,1,390,8,MCAV,NaN,...,Core_EtOH,13,Diseased_Tissue,NaN,052022_BEL_CBC_T1_13_MCAV,Depleted_UML_NARWHAL_R1_B3,DNA_extracted,sequenced,Only sand in tissue sample,T1_8_MCAV
929,62019,BEL,CBC,6/24/19,CBC30N,1,390,8,MCAV,NaN,...,Core_EtOH,16,Healthy,mucus sheaths,062019_BEL_CBC_T1_16_MCAV,UML_NARWHAL_R1_B1,DNA_extracted,sequenced,NaN,T1_8_MCAV


##### **PSTR**

In [44]:
# crosscheck 2019 and 2022 colonies and samples

# pstr subset 
pstr=matched_metadata[matched_metadata['Species']=="PSTR"]
# list of all samples in 2019
meta_2019=pstr[
    (pstr['Month_year']==62019) |
    (pstr['Month_year']==102019)]
ids_2019=set(meta_2019['colony_id'].unique())
# list of all samples in 2022
meta_2022=pstr[
    (pstr['Month_year']==52022) |
    (pstr['Month_year']==122022)]
ids_2022=set(meta_2022['colony_id'].unique())
# do they match 
ids_2019 == ids_2022

False

In [45]:
# total pstr colonies 
print(len(pstr['colony_id'].unique()))

# colonies present in both years
print(len(ids_2019 & ids_2022))
# 11 shared

print(len(ids_2019))
# 21 initial

print(len(ids_2022))
# 10 added, 10 died in 5/22

31
11
21
21


In [46]:
# confirming that there 10 added and 10 died - yes 

# colonies not in 2022 
print('colonies not in 2022:',len(ids_2019 - ids_2022), 'colonies',
      ids_2019 - ids_2022)
# manually checking fate 
# all died in 052022

# new colonies in 2022 not present in 2019
print('new colonies in 2022 not present in 2019:',len(ids_2022 - ids_2019), 'colonies',
      ids_2022 - ids_2019)
# manually checking fate 
# all tagged in 2022

colonies not in 2022: 10 colonies {'T3_32_PSTR', 'T3_30_PSTR', 'T1_422_PSTR', 'T2_403_PSTR', 'T1_417_PSTR', 'T3_11_PSTR', 'T3_16_PSTR', 'T1_419_PSTR', 'T2_427_PSTR', 'T1_404_PSTR'}
new colonies in 2022 not present in 2019: 10 colonies {'T4_96_PSTR', 'T4_97_PSTR', 'T4_80_PSTR', 'T3_75_PSTR', 'T3_70_PSTR', 'T4_98_PSTR', 'T2_28_PSTR', 'T4_79_PSTR', 'T2_32_PSTR', 'T3_74_PSTR'}


In [47]:
# check healthy 

# healthy 2019 samples 
healthy_pstr2019=meta_2019[meta_2019['Health_status']=='Healthy']
ids_h2019=set(healthy_pstr2019['colony_id'].unique())

# healthy 2022 samples 
healthy_pstr2022=meta_2022[meta_2022['Health_status']=='Healthy']
ids_h2022=set(healthy_pstr2022['colony_id'].unique())

# do they match 
ids_h2019 == ids_h2022

False

In [48]:
# colonies present in both years
print(len(ids_h2019 & ids_h2022))

print(len(ids_h2019))

print(len(ids_h2022))

9
21
16


##### **PAST**

In [49]:
# crosscheck 2019 and 2022 colonies and samples
# no past colonies were added in 2022 so why are there 24 healthy in 2022 and 20 healthy in 2019? 

# past subset 
past=matched_metadata[matched_metadata['Species']=="PAST"]
# list of all samples in 2019
meta_2019=past[
    (past['Month_year']==62019) |
    (past['Month_year']==102019)]
ids_2019=set(meta_2019['colony_id'].unique())
# list of all samples in 2022
meta_2022=past[
    (past['Month_year']==52022) |
    (past['Month_year']==122022)]
ids_2022=set(meta_2022['colony_id'].unique())
# do they match 
ids_2019 == ids_2022

False

In [50]:
# total past colonies 
print(len(past['colony_id'].unique()))

# colonies present in both years
print(len(ids_2019 & ids_2022))
# 16 shared

print(len(ids_2019))
# 20 initial

print(len(ids_2022))
# none added, lost 4
# 9 colonies had a sample in 52022 and 122022 
16+9 
# there is one more sample that should be there if 9 colonies got disease ..
# -> solved based on for loop below: missing a disease margin from one of the colonies in 2022

20
16
20
16


25

In [51]:
# confirming - no colonies added, 2 died, and 2 weren't sampled again for some reason

# colonies not in 2022 
print('colonies not in 2022:',len(ids_2019 - ids_2022), 'colonies',
      ids_2019 - ids_2022)
# manually checking fate 
# 347 and 12flag died 
# see below about t2_47 and t3_13

# new colonies in 2022 not present in 2019
print('new colonies in 2022 not present in 2019:',len(ids_2022 - ids_2019), 'colonies',
      ids_2022 - ids_2019)
# none added in 2022

colonies not in 2022: 4 colonies {'T3_13_PAST', 'T2_47_PAST', 'T2_347_PAST', 'T3_12flag_PAST'}
new colonies in 2022 not present in 2019: 0 colonies set()


In [52]:
# weird ones: 
past[past['colony_id']=='T2_47_PAST']
# only a sample from 2019 but never has a mortality date 

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
915,62019,BEL,CBC,6/25/19,SR30N,2,47,47,PAST,NaN,...,Core_EtOH,29,Healthy,orange tag,062019_BEL_CBC_T2_29_PAST,UML_NARWHAL_R1_B1,DNA_extracted,NaN,NaN,T2_47_PAST


In [53]:
# weird ones: 
past[past['colony_id']=='T3_13_PAST']
# only a sample from 2019 but never has a mortality date 

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
967,62019,BEL,CBC,6/23/19,Lagoon,3,358,13,PAST,NaN,...,Core_EtOH,10,Healthy,NaN,062019_BEL_CBC_T3_10_PAST,UML_NARWHAL_R1_B1,penguin,NaN,NaN,T3_13_PAST


In [54]:
# weird ones: 
past[past['colony_id']=='T3_24_PAST']
# check that this healthy 12/2022 sample is correct? 
# colony data says this colony died in 5/22 but health statuses say healthy through 12/2022

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
431,122022,BEL,CBC,12/3/22,Lagoon,3,NaN,24,PAST,NaN,...,Core_EtOH,122,Healthy,NaN,122022_BEL_CBC_T3_122_PAST,UML_NARWHAL_R1_B5,UML_NARWHAL_R2_B29,NaN,NaN,T3_24_PAST
995,62019,BEL,CBC,6/23/19,Lagoon,3,314,24,PAST,NaN,...,Core_EtOH,23,Healthy,NaN,062019_BEL_CBC_T3_23_PAST,UML_NARWHAL_R1_B1,penguin,NaN,NaN,T3_24_PAST


In [55]:
# check healthy 

# healthy 2019 samples 
healthy_past2019=meta_2019[meta_2019['Health_status']=='Healthy']
ids_h2019=set(healthy_past2019['colony_id'].unique())

# healthy 2022 samples 
healthy_past2022=meta_2022[meta_2022['Health_status']=='Healthy']
ids_h2022=set(healthy_past2022['colony_id'].unique())

# do they match 
ids_h2019 == ids_h2022

False

In [56]:
# colonies present in both years
print(len(ids_h2019 & ids_h2022))
# 5 didn't stay healthy 

print(len(ids_h2019))

print(len(ids_h2022))

15
20
15


In [57]:
print('healthy colonies that were not healthy in 2022:',len(ids_h2019 - ids_h2022), 'colonies',
      ids_h2019 - ids_h2022)
print('only difference b/w above difference and this healthy list:', ids_2022-ids_h2022)
# manually checking fate 

# new healthy colonies in 2022 not present in 2019
print('new healthy colonies in 2022 not present in 2019:',len(ids_h2022 - ids_h2019), 'colonies',
      ids_h2022 - ids_h2019)

healthy colonies that were not healthy in 2022: 5 colonies {'T2_347_PAST', 'T3_12flag_PAST', 'T2_47_PAST', 'T3_18_PAST', 'T3_13_PAST'}
only difference b/w above difference and this healthy list: {'T3_18_PAST'}
new healthy colonies in 2022 not present in 2019: 0 colonies set()


In [58]:
# weird ones: 
past[past['colony_id']=='T3_18_PAST']
# diseased 

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
424,122022,BEL,CBC,12/3/22,Lagoon,3,NaN,18,PAST,NaN,...,Core_EtOH,115,Diseased_Tissue,NaN,122022_BEL_CBC_T3_115_PAST,UML_NARWHAL_R1_B5,UML_NARWHAL_R2_B26,NaN,NaN,T3_18_PAST
438,122022,BEL,CBC,12/3/22,Lagoon,3,NaN,18,PAST,NaN,...,Core_EtOH,129,Diseased_Margin,NaN,122022_BEL_CBC_T3_129_PAST,UML_NARWHAL_R1_B5,UML_NARWHAL_R2_B26,NaN,NaN,T3_18_PAST
786,52022,BEL,CBC,5/20/22,Lagoon,3,365,18,PAST,NaN,...,Core_EtOH,54,Diseased_Margin,"TL (F, SA)",052022_BEL_CBC_T3_54_PAST,UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B26,NaN,NaN,T3_18_PAST
787,52022,BEL,CBC,5/20/22,Lagoon,3,365,18,PAST,NaN,...,Core_EtOH,55,Diseased_Tissue,"TL (F, SA)",052022_BEL_CBC_T3_55_PAST,UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B26,NaN,NaN,T3_18_PAST
971,62019,BEL,CBC,6/23/19,Lagoon,3,365,18,PAST,NaN,...,Core_EtOH,12,Healthy,NaN,062019_BEL_CBC_T3_12_PAST,UML_NARWHAL_R1_B1,UML_NARWHAL_R2_B29,NaN,NaN,T3_18_PAST


In [59]:
# so only 1 2019 healthy colony got disease in 2022? 
# so why is there 4 more healthy samples in 2022 than in 2019??
past_meta=matched_metadata[matched_metadata['Species']=='PAST']
len(past_meta)
# 53 total samples 

53

In [60]:
past_metah=past_meta[past_meta['Health_status']=="Healthy"]
print(len(past_metah))
# 44 total healthy samples 
len(past_metah['colony_id'].unique())
# 20 unique colonies 
# do some have duplicate healthy samples in 2022??

44


20

In [61]:
# sep 5 and 122022 - are there any colony IDs that have samples from both dates? 
pasth_52022=healthy_past2022[(healthy_past2022['Month_year']==52022)]
pasth_122022=healthy_past2022[(healthy_past2022['Month_year']==122022)]

ids_52022=set(pasth_52022['colony_id'].unique())
ids_2022=set(pasth_122022['colony_id'].unique())

print('healthy colonies sampled in BOTH 5/22 and 12/22:',len(ids_52022 & ids_2022),ids_52022 & ids_2022)

healthy colonies sampled in BOTH 5/22 and 12/22: 9 {'T2_63_PAST', 'T3_10_PAST', 'T3_8_PAST', 'T1_13_PAST', 'T1_21_PAST', 'T2_68_PAST', 'T2_57_PAST', 'T3_6_PAST', 'T1_2_PAST'}


In [62]:
past[past['colony_id']=='T1_13_PAST']

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
310,122022,BEL,CBC,12/2/22,CBC30N,1,NaN,13,PAST,NaN,...,Core_EtOH,152,Healthy,NaN,122022_BEL_CBC_T1_152_PAST,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B29,NaN,NaN,T1_13_PAST
713,52022,BEL,CBC,5/21/22,CBC30N,1,395,13,PAST,NaN,...,Core_EtOH,53,Healthy,NaN,052022_BEL_CBC_T1_53_PAST,UML_NARWHAL_R1_B3,penguin,NaN,NaN,T1_13_PAST
921,62019,BEL,CBC,6/24/19,CBC30N,1,395,13,PAST,NaN,...,Core_EtOH,12,Healthy,NaN,062019_BEL_CBC_T1_12_PAST,UML_NARWHAL_R1_B1,penguin,NaN,NaN,T1_13_PAST


In [63]:
past[past['colony_id']=='T1_20_PAST']

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,...,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony_id
263,122022,BEL,CBC,12/4/22,CBC30N,1,NaN,20,PAST,NaN,...,Core_EtOH,134,Diseased_Margin,NaN,122022_BEL_CBC_T1_134_PAST,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B26,NaN,NaN,T1_20_PAST
267,122022,BEL,CBC,12/2/22,CBC30N,1,NaN,20,PAST,NaN,...,Core_EtOH,139,Diseased_Tissue,NaN,122022_BEL_CBC_T1_139_PAST,UML_NARWHAL_R1_B4,UML_NARWHAL_R2_B26,NaN,NaN,T1_20_PAST
724,52022,BEL,CBC,5/21/22,CBC30N,1,386,20,PAST,NaN,...,Core_EtOH,61,Healthy,NaN,052022_BEL_CBC_T1_61_PAST,UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B29,NaN,NaN,T1_20_PAST
933,62019,BEL,CBC,6/24/19,CBC30N,1,386,20,PAST,NaN,...,Core_EtOH,18,Healthy,NaN,062019_BEL_CBC_T1_18_PAST,UML_NARWHAL_R1_B1,penguin,NaN,NaN,T1_20_PAST


### Match with colony data
- crosscheck colony conditions and sample data 

In [64]:
# upload and clean up colony data 
colony=pd.read_csv('//project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/metadata/CBC_ColonyData.csv')
colony = colony.iloc[:, 1:]

# add colony ID - t# newtagnum species
colony = colony.copy()
colony['TransectNum_str'] = 'T' + colony['TransectNum'].astype(str)
colony['colony_id'] = colony[['TransectNum_str', 'NewTagNum', 'Species']].astype(str).agg('_'.join, axis=1)
colony.drop(columns='TransectNum_str', inplace=True)

# remove bb and hangman
colony=colony[
    (colony['Transect']!='BB') &
    (colony['Transect']!='HANGMAN')]

In [65]:
# create function to crosscheck colony health with samples 

# if colony_condition is healthy sample = healthy 
# if condition is diseased healthy = diseased_margin and diseased_tissue (but rn these are on 2 diff rows)
# if condition is Not_visited, sample = NaN
# if condition is Dead, sample = NaN 
def status_match(group):
    # get the condition and list sample statuses of each colony at each monthyear (by row) 
    cond = group['colony_condition'].iloc[0]
    statuses = group['Health_status'].dropna().tolist()

    # match colony conditions to sample conditions 
    if cond == 'Healthy':
        return all(s == 'Healthy' for s in statuses)
    elif cond == 'Diseased':
        return all(x in statuses for x in ['Diseased_Tissue', 'Diseased_Margin'])
    elif cond in ['Dead', 'Not_Visited']:
        return all(pd.isna(s) for s in group['Health_status'])
        
    else:
        return False

In [66]:
# create for loop for each specie (of my samples) 
species_list=matched_metadata['Species'].unique().tolist()
species_list

species_checks = {}

for specie in species_list: 
    # filter colony and sample data for each specie 
    filtered_colony=colony[colony['Species']==specie]

    # reduce cols and pivot 
    conditions_long = (
        filtered_colony.loc[:, ['colony_id', '062019_Condition', '052022_Condition', '122022_Condition']]
                   .melt(id_vars='colony_id', var_name='Month_year', value_name='colony_condition')
    )
    conditions_long['Month_year'] = conditions_long['Month_year'].str.extract(r'(\d+)').astype(int)
   
    # repeat for sample data 
    filtered_samples=matched_metadata[matched_metadata['Species']==specie]
    filtered_samples=filtered_samples.loc[:,('colony_id','Month_year','Health_status','Sampling_notes','Tubelabel_species')] 
    filtered_samples['sample_condition']=filtered_samples['Health_status']
    
    # merge dfs 
    merged = pd.merge(
    conditions_long.merge(filtered_colony, on='colony_id', how='left'),
    filtered_samples,
    on=['colony_id', 'Month_year'],
    how='outer'
    )
    
    checks = []
        # check each 'group' (unique combos of colony and monthyear) and store results 
    for (colony_id, month), group in merged.groupby(['colony_id', 'Month_year']):
        checks.append({
            'colony_id': colony_id,
            'Date_InitialTag': group['Date_InitialTag'].iloc[0],
            'Month_year': month,
                
            # run function &
            # create col containing results of the function that matches statuses
            'Match': status_match(group),
                
            'colony_condition': group['colony_condition'].iloc[0],
            'sample_statuses': group['Health_status'].tolist(),
            'sample_ids': group['Tubelabel_species'].tolist(),
            'mortality_date' : group['Date_DocumentedMortality'].iloc[0]
    })
    # store sp checks in dict
    species_checks[specie]=pd.DataFrame(checks)
    

In [67]:
# find mismatches for each specie and view

false_matches = {}
for specie in species_list:
    df = pd.DataFrame(species_checks[specie])
    false = df[(df['Match'] == False) & (~df['colony_condition'].isna())]
    false_matches[specie] = false

# combine
all_false_matches = pd.concat(
    [df.assign(Species=specie) for specie, df in false_matches.items()],
    ignore_index=True
)

all_false_matches

,colony_id,Date_InitialTag,Month_year,Match,colony_condition,sample_statuses,sample_ids,mortality_date,Species
0,T1_23_OANN,5/21/22,52022.0,False,DC,[Healthy],[052022_BEL_CBC_T1_35_OANN],Healthy,OANN
1,T4_99_OANN,12/5/22,122022.0,False,Diseased,[nan],[nan],Diseased,OANN
2,T1_19_PAST,6/21/19,52022.0,False,Diseased,[Diseased_Tissue],[052022_BEL_CBC_T1_52_PAST],4/1/24,PAST
3,T1_8_MCAV,6/24/19,122022.0,False,Diseased,[Diseased_Tissue],[122022_BEL_CBC_T1_144_MCAV],9/25/23,MCAV
4,T2_59_MCAV,6/21/19,52022.0,False,Diseased,[nan],[nan],Diseased,MCAV
5,T3_67_MCAV,12/3/22,122022.0,False,Diseased,[Diseased_Tissue],[122022_BEL_CBC_T3_145_MCAV],9/25/23,MCAV
6,T3_71_MCAV,12/3/22,122022.0,False,DC,[Healthy],[122022_BEL_CBC_T3_155_MCAV],Healthy,MCAV


In [68]:
past=species_checks['PAST']

In [69]:
# Create a wide-format view for colony statuses
colony_pivot = past.pivot(index='colony_id', columns='Month_year', values='colony_condition')
colony_pivot['type'] = 'Colony'

# Create a wide-format view for sample statuses
sample_pivot = past.pivot(index='colony_id', columns='Month_year', values='sample_statuses')
sample_pivot['type'] = 'Samples'

# Combine both
combined = pd.concat([colony_pivot, sample_pivot])
combined = combined.sort_values(['colony_id', 'type'])  # Group colony + samples together

# Reorder the date columns explicitly
ordered_cols = ['colony_id', 'type', 62019, 52022, 122022]
combined = combined[[col for col in ordered_cols if col in combined.columns]]


In [70]:
combined
# each 2019 colony gets 2 healthy samples in 2022 unless noted otherwise (not incl dead): 
    # t1_19 1 disease in 52022, 1 healthy in 122022 
    # t1_20 1 healthy in 52022, 2 disease in 122022
    # t2_56 2 disease in 52022, 1 healthy in 122022 
    # t3 24 no 52022 sample 
    # t3 34 no 52022 sample 
    # t3 7 no 52022 sample 
# so each 

# 1 full disease - t3 18

# 5 - no samples in 5 and 122022


Month_year,type,62019,52022,122022
colony_id,,,,
T1_13_PAST,Colony,Healthy,Healthy,Healthy
T1_13_PAST,Samples,[Healthy],[Healthy],[Healthy]
T1_19_PAST,Colony,Healthy,Diseased,Healthy
T1_19_PAST,Samples,[Healthy],[Diseased_Tissue],[Healthy]
T1_20_PAST,Colony,Healthy,Healthy,Diseased
T1_20_PAST,Samples,[Healthy],[Healthy],"[Diseased_Margin, Diseased_Tissue]"
T1_21_PAST,Colony,Healthy,Healthy,Healthy
T1_21_PAST,Samples,[Healthy],[Healthy],[Healthy]
T1_2_PAST,Colony,Healthy,Healthy,Healthy


In [71]:
mcav=species_checks['MCAV']

In [72]:
# Create a wide-format view for colony statuses
colony_pivot = mcav.pivot(index='colony_id', columns='Month_year', values='colony_condition')
colony_pivot['type'] = 'Colony'

# Create a wide-format view for sample statuses
sample_pivot = mcav.pivot(index='colony_id', columns='Month_year', values='sample_statuses')
sample_pivot['type'] = 'Samples'

# Combine both
combined = pd.concat([colony_pivot, sample_pivot])
combined = combined.sort_values(['colony_id', 'type'])  # Group colony + samples together

# Reorder the date columns explicitly
ordered_cols = ['colony_id', 'type', 62019, 52022, 122022]
combined = combined[[col for col in ordered_cols if col in combined.columns]]

In [73]:
combined.shape
64/2

32.0

In [74]:
combined.head(30)
# diseased/dead
# 9 

# h -> h
# extra h sample from t1_24 in 5/22
# t2 53, t2 59 missing sample 5/22

Month_year,type,62019,52022,122022
colony_id,,,,
T1_14_MCAV,Colony,Healthy,Diseased,Dead
T1_14_MCAV,Samples,[Healthy],"[Diseased_Margin, Diseased_Tissue]",[nan]
T1_15_MCAV,Colony,Healthy,Diseased,Dead
T1_15_MCAV,Samples,[Healthy],"[Diseased_Margin, Diseased_Tissue]",[nan]
T1_24_MCAV,Colony,Healthy,Healthy,Healthy
T1_24_MCAV,Samples,[Healthy],"[Healthy, Healthy]",[Healthy]
T1_329_MCAV,Colony,Healthy,Dead,Not_Visited
T1_329_MCAV,Samples,[Healthy],[nan],[nan]
T1_333_MCAV,Colony,Healthy,Dead,Not_Visited


In [75]:
combined.tail(34)
# diseased/dead
# 8 
# t3 67 missing margin sample in 122022

# h -> h
# t2 61 - died in 122022 (no sample) 
# 4 healthy added in 122022?

Month_year,type,62019,52022,122022
colony_id,,,,
T2_61_MCAV,Colony,Healthy,Healthy,Dead
T2_61_MCAV,Samples,[Healthy],[Healthy],[nan]
T2_69_MCAV,Colony,Healthy,Healthy,Healthy
T2_69_MCAV,Samples,[Healthy],[Healthy],[Healthy]
T3_12_MCAV,Colony,Healthy,Dead,Dead
T3_12_MCAV,Samples,[Healthy],[nan],[nan]
T3_14_MCAV,Colony,Healthy,Healthy,Healthy
T3_14_MCAV,Samples,[Healthy],[Healthy],[Healthy]
T3_15_MCAV,Colony,Healthy,Diseased,Diseased


In [76]:
# 24 colonies minus full diseased and dead 
print(24-13)
# 12 colonies left 
# and 4 new in 122022 
    # minus 3 that are missing a sample 
    # plus 1 that has an extra sample
print((11-3)*2)
18+4+3+1
#idk whats happening 

11
16


26

In [77]:
condensed

,Year,Health_status,MCAV,MMEA,OANN,OFAV,PAST,PSTR
0,2019,Healthy,24,13,0,0,20,21
1,2022,Diseased_Margin,9,0,2,2,4,6
2,2022,Diseased_Tissue,11,0,3,2,5,6
3,2022,Healthy,24,0,8,15,24,21


In [78]:
# Reorder the date columns explicitly
ordered_cols = ['colony_id', 62019, 52022, 122022,'type']
sample_pivot = sample_pivot[[col for col in ordered_cols if col in sample_pivot.columns]]

print(sample_pivot.shape)
sample_pivot

# count num of healthy in 62019 


(32, 4)


Month_year,62019,52022,122022,type
colony_id,,,,
T1_14_MCAV,[Healthy],"[Diseased_Margin, Diseased_Tissue]",[nan],Samples
T1_15_MCAV,[Healthy],"[Diseased_Margin, Diseased_Tissue]",[nan],Samples
T1_24_MCAV,[Healthy],"[Healthy, Healthy]",[Healthy],Samples
T1_329_MCAV,[Healthy],[nan],[nan],Samples
T1_333_MCAV,[Healthy],[nan],[nan],Samples
T1_342_MCAV,[Healthy],[nan],[nan],Samples
T1_355_MCAV,[Healthy],[nan],[nan],Samples
T1_7_MCAV,[Healthy],[Healthy],[Healthy],Samples
T1_8_MCAV,[Healthy],"[Diseased_Margin, Diseased_Tissue]",[Diseased_Tissue],Samples


In [79]:
pstr=species_checks['PSTR']

# Create a wide-format view for colony statuses
colony_pivot = pstr.pivot(index='colony_id', columns='Month_year', values='colony_condition')
colony_pivot['type'] = 'Colony'

# Create a wide-format view for sample statuses
sample_pivot = pstr.pivot(index='colony_id', columns='Month_year', values='sample_statuses')
sample_pivot['type'] = 'Samples'

# Combine both
combined = pd.concat([colony_pivot, sample_pivot])
combined = combined.sort_values(['colony_id', 'type'])  # Group colony + samples together

# Reorder the date columns explicitly
ordered_cols = ['colony_id', 'type', 62019, 52022, 122022]
combined = combined[[col for col in ordered_cols if col in combined.columns]]

# remove if na across all cols 
combined_cleaned = combined.dropna(how='all')

In [80]:
combined_cleaned.head(30)

Month_year,type,62019.0,52022.0,122022.0
colony_id,,,,
T1_12_PSTR,Colony,Healthy,Healthy,Healthy
T1_12_PSTR,Samples,[nan],[Healthy],[Healthy]
T1_404_PSTR,Colony,Healthy,Dead,Not_Visited
T1_404_PSTR,Samples,[nan],[nan],[nan]
T1_417_PSTR,Colony,Healthy,Dead,Dead
T1_417_PSTR,Samples,[nan],[nan],[nan]
T1_419_PSTR,Colony,Healthy,Dead,Not_Visited
T1_419_PSTR,Samples,[nan],[nan],[nan]
T1_422_PSTR,Colony,Healthy,Dead,Not_Visited


In [81]:
# any rows that have all nas in samples? 